# 第 5 天 – 人工智能宣传册生成器

## 练习目标（理念）

这个笔记本为初创公司构建一个**人工智能驱动的投资者宣传册（brochure）生成器**。

系统流水线大致是：

1. 从公司网站上抓取链接（Web Scraping）
2. 用 LLM 过滤出与投资者相关的页面
3. 生成结构化的启动/投资手册
4. （可选）将手册翻译成另一种语言
5. 用**流式输出（streaming）**获得交互式体验

对应课程 **Week 1 Day 5**：多步 LLM 调用 + 网页抓取 + 结构化输出。

## 怎么跑

1. 准备好 `.env`（含 OpenAI API Key）
2. 从上到下依次运行每个单元格（Shift+Enter）
3. 在「用户输入」格子里改 `site_url` / `target_language`


## 架构概述

该系统遵循模块化流水线（pipeline）：

```
网站
  ↓
链接抓取（scrape_links）
  ↓
LLM 链接过滤（filter_links）
  ↓
宣传册生成（generate_brochure，可流式）
  ↓
可选翻译（translate_brochure）
```

每个阶段都写成单独函数，方便以后替换抓取器、换模型或加缓存。


## 安装依赖项

本笔记本需要这些库（若环境已装可跳过）：

- `requests`：网页抓取（HTTP GET）
- `BeautifulSoup`（`bs4`）：HTML 解析
- `openai`：调用 Chat Completions API
- `python-dotenv`：从 `.env` 加载密钥


In [ ]:
# ========== 导入与客户端初始化 ==========

# 导入 requests：用 HTTP 请求抓取网页 HTML
import requests
# 从 bs4 导入 BeautifulSoup：把 HTML 解析成可查询的树
from bs4 import BeautifulSoup
# 从 urllib.parse 导入 urljoin：把相对链接拼成绝对 URL
from urllib.parse import urljoin
# 导入 json：把链接列表序列化进 prompt（便于 LLM 阅读）
import json
# 导入 os：读环境变量（本格主要给 dotenv / OpenAI 间接使用）
import os

# 从 dotenv 导入 load_dotenv、find_dotenv：定位并加载 .env，避免把密钥写进代码
from dotenv import load_dotenv, find_dotenv
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI

# 加载环境变量：find_dotenv() 向上找 .env，再读入进程环境
load_dotenv(find_dotenv())

# 初始化 OpenAI 客户端：默认从环境变量 OPENAI_API_KEY 取密钥
client = OpenAI()

# 选用的模型名（Model ID）：后面所有 chat.completions.create 都引用这个常量
MODEL = "gpt-5-nano"


## 用户输入

运行管道前，用户需要提供：

- **网站网址** `site_url`：要分析的公司主页
- **可选翻译语言** `target_language`：设为 `None` 则保持英文输出


In [ ]:
# ========== 可调参数：目标网站与翻译语言 ==========

# 公司官网入口：抓取与分析都从这里开始（可改成你关心的公司）
site_url = "https://edwarddonner.com/"

# 目标翻译语言：字符串传给翻译 prompt；设为 None 则跳过翻译、保留英文
target_language = "Spanish"   # None for English output


## 步骤 1 – 抓取网站链接

`scrape_links(base_url)` 会：

1. 用 `requests.get` 拉取网页 HTML
2. 用 BeautifulSoup 找出所有带 `href` 的锚标签（`<a>`）
3. 用 `urljoin` 把相对路径转成绝对 URL
4. 用 `set` 去重后返回链接列表


In [ ]:
# ========== 步骤 1：从首页 HTML 收集外链 ==========

def scrape_links(base_url):
    # GET 目标页面：response.text 是原始 HTML 字符串
    response = requests.get(base_url)
    # 用 html.parser 解析 DOM，方便 find_all 查询标签
    soup = BeautifulSoup(response.text, "html.parser")

    # 用集合去重：同一链接可能在页脚/导航重复出现
    links = set()

    # 遍历所有带 href 属性的 <a> 标签
    for tag in soup.find_all("a", href=True):

        # 取出原始 href（可能是相对路径，如 /about）
        href = tag["href"]
        # 相对路径 + base_url → 绝对 URL，便于后续再抓取
        absolute = urljoin(base_url, href)

        # 只保留 http/https 链接，丢掉 mailto:、javascript: 等
        if absolute.startswith("http"):
            links.add(absolute)

    # 转成 list：后面要 json.dumps 进 prompt
    return list(links)


## 步骤 2 – 使用 LLM 过滤相关链接

网站上很多链接对**投资者手册**没用（隐私政策、登录页等）。

这里用 LLM 只保留相关链接，例如：

- 产品页面
- 技术概述
- 公司概况
- 案例研究

system prompt 要求模型返回 JSON，便于后续程序消费。


In [ ]:
# ========== 链接过滤：system prompt（发给模型的角色指令，保持英文） ==========

# 字符串内容是 prompt，翻译会改变模型行为，故保留英文原文
LINK_FILTER_SYSTEM_PROMPT = """
You are an AI assistant helping create an investor brochure.

From a list of website links, return ONLY links useful for understanding the company.

Include:
- product pages
- technology explanation
- company overview
- pricing
- solutions
- case studies

Exclude:
- privacy policy
- terms
- cookies
- login
- careers
- legal pages

Return JSON:

{
 "relevant_links":[
   {"url":"...","reason":"..."}
 ]
}
"""


In [ ]:
# ========== 步骤 2：调用 LLM，从全量链接里筛出相关链接 ==========

def filter_links(links):

    # 拼 user prompt：把链接列表以缩进 JSON 形式塞进消息
    user_prompt = f"""
Analyze the following links and return only relevant ones.

Links:
{json.dumps(links, indent=2)}
"""

    # Chat Completions：system 定规则，user 给具体链接列表
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":LINK_FILTER_SYSTEM_PROMPT},
            {"role":"user","content":user_prompt}
        ]
    )

    # 取出第一条回复的文本（期望是 JSON 字符串）
    return response.choices[0].message.content


## 步骤 3 – 生成投资者手册

有了过滤后的链接，模型会生成面向初创投资者的结构化手册（产品、市场、商业模式等）。


In [ ]:
# ========== 宣传册生成：system prompt（角色与章节结构，保持英文） ==========

BROCHURE_SYSTEM_PROMPT = """
You are an expert startup analyst.

Generate a professional investor brochure.

Include sections:

Product Overview
Target Market
Business Model
Technology
Key Advantages

Use concise professional language.
"""


## 流式输出（Streaming）

为了让笔记本更有「边生成边显示」的交互感，后面用 `stream=True`，按 token / chunk 打印，而不是等整段生成完。


In [ ]:
# ========== 通用流式调用：边收 delta 边 print，最后返回完整文本 ==========

def stream_response(system_prompt, user_prompt):

    # stream=True：服务端持续推送增量；返回可迭代的 stream 对象
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":system_prompt},
            {"role":"user","content":user_prompt}
        ],
        stream=True
    )

    # 累积完整输出，供后续翻译或落盘
    output=""

    # 逐块读取流式响应
    for chunk in stream:

        # delta.content：本块新增的文本；结束块可能是 None
        token = chunk.choices[0].delta.content

        if token:
            # end="" 不换行；flush=True 立刻刷到终端，看起来像打字机
            print(token, end="", flush=True)
            output += token

    # 流结束后补一个空行，方便阅读
    print("\n")

    return output


In [ ]:
# ========== 步骤 3：用过滤后的链接生成投资者宣传册 ==========

def generate_brochure(filtered_links):

    # user prompt：把相关链接交给模型当「素材」
    user_prompt = f"""
Generate an investor brochure using the following company links:

{filtered_links}
"""

    # 复用流式助手：system 定结构，user 给素材
    return stream_response(
        BROCHURE_SYSTEM_PROMPT,
        user_prompt
    )


## 步骤 4 – 翻译手册

若设置了 `target_language`，可把生成的手册翻译给国际投资者阅读。


In [ ]:
# ========== 翻译：system prompt（保持英文，避免改变翻译风格指令） ==========

TRANSLATION_SYSTEM_PROMPT = """
You are a professional business translator.

Translate the brochure into the requested language.

Preserve formatting and tone.
"""


In [ ]:
# ========== 步骤 4：把宣传册流式翻译成目标语言 ==========

def translate_brochure(text, language):

    # user prompt：嵌入目标语言名与原文
    user_prompt = f"""
Translate the following brochure into {language}:

{text}
"""

    # 同样走流式打印，便于边看边等
    return stream_response(
        TRANSLATION_SYSTEM_PROMPT,
        user_prompt
    )


## 步骤 5 – 运行完整管道

按顺序：抓链接 → LLM 过滤 → 生成手册 →（可选）翻译 → 在笔记本里用 Markdown 展示，并写一份 HTML 备份。


In [ ]:
# ========== 端到端：串联整条宣传册流水线并落盘 ==========

# 1) 从官网抓取全部 http(s) 链接
links = scrape_links(site_url)

# 2) 让 LLM 只保留对投资者有用的链接（返回文本，通常是 JSON）
filtered_links = filter_links(links)

# 3) 基于过滤结果流式生成英文宣传册
brochure = generate_brochure(filtered_links)

# 4) 若设置了目标语言，再流式翻译（会覆盖 brochure 变量）
if target_language:
    brochure = translate_brochure(brochure, target_language)

# 在 Jupyter 里把 Markdown 渲染成富文本预览
from IPython.display import Markdown, display
display(Markdown(brochure))

# 包一层最简 HTML，方便用浏览器打开查看
html_content = f"<html><body><pre>{brochure}</pre></body></html>"

# 写入当前工作目录下的 brochure_output.html（utf-8 避免多语言乱码）
with open("brochure_output.html", "w", encoding="utf-8") as f:
    f.write(html_content)
